# Neuron from Scratch using NumPy

## Objective

In this notebook, we will build a **single perceptron (neuron) from scratch using NumPy**.

We will teach it to solve the **AND gate**, a simple linearly separable problem.

### What we will learn

1. What a neuron does
2. Inputs, weights, and bias
3. Weighted sum
4. Step activation function
5. Prediction
6. Error calculation
7. Weight and bias updates
8. Training over multiple epochs
9. Testing the trained neuron
10. Why a single perceptron cannot solve XOR

### Core formula

A neuron first calculates:

`z = w1*x1 + w2*x2 + b`

Then applies an activation function:

`prediction = step(z)`

The perceptron learns by adjusting its **weights and bias** when its prediction is incorrect.

## 1. Import NumPy

NumPy provides arrays and mathematical operations that make it easy to implement the neuron.

We are intentionally **not using TensorFlow/Keras** here because the goal is to understand the mechanics of a neuron ourselves.

In [1]:
import numpy as np

## 2. Create the AND Gate Dataset

The AND gate has two inputs and one expected output.

| x1 | x2 | Expected output |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

Only when **both inputs are 1** should the output be 1.

`X` contains the input examples.

`y` contains the correct answers (labels) that the neuron should learn.

In [2]:
# Input data
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

# Expected outputs
y = np.array([0, 0, 0, 1])

print("Input X:")
print(X)

print("\nExpected output y:")
print(y)

Input X:
[[0 0]
 [0 1]
 [1 0]
 [1 1]]

Expected output y:
[0 0 0 1]


## 3. Initialize Weights, Bias, and Training Settings

A neuron has:

- **Weights** → determine how strongly each input influences the result.
- **Bias** → shifts the decision boundary.
- **Learning rate** → controls how large each update is.
- **Epochs** → number of times the complete training dataset is processed.

We start the weights and bias at zero.

The neuron will gradually change them during training.

In [3]:
# One weight for each input
weights = np.zeros(2)

# Initial bias
bias = 0.0

# How quickly the neuron learns
learning_rate = 0.1

# Number of complete passes through the training data
epochs = 10

print("Initial weights:", weights)
print("Initial bias:", bias)
print("Learning rate:", learning_rate)
print("Epochs:", epochs)

Initial weights: [0. 0.]
Initial bias: 0.0
Learning rate: 0.1
Epochs: 10


## 4. Create the Step Activation Function

The perceptron uses a simple **step function**.

It converts the weighted sum into either `0` or `1`.

- If `z > 0` → output `1`
- Otherwise → output `0`

This makes the neuron behave like a binary classifier.

In [4]:
def step_function(z):
    if z > 0:
        return 1
    return 0


# Test the activation function
print("step(-2) =", step_function(-2))
print("step(3)  =", step_function(3))

step(-2) = 0
step(3)  = 1


## 5. Understand the Weighted Sum

For every input, the neuron calculates:

`z = np.dot(inputs, weights) + bias`

For two inputs this is equivalent to:

`z = x1*w1 + x2*w2 + bias`

### Example

If:

- `inputs = [1, 1]`
- `weights = [0.5, 0.5]`
- `bias = -0.5`

then:

`z = (1*0.5) + (1*0.5) - 0.5 = 0.5`

The step function then converts `0.5` into `1`.

In [5]:
# Small example of the weighted sum

example_inputs = np.array([1, 1])
example_weights = np.array([0.5, 0.5])
example_bias = -0.5

z = np.dot(example_inputs, example_weights) + example_bias

print("Weighted sum:", z)
print("Prediction:", step_function(z))

Weighted sum: 0.5
Prediction: 1


## 6. Train the Perceptron

Now we implement the learning algorithm.

For every training example:

### Step 1 — Calculate weighted sum

`z = x·w + b`

### Step 2 — Make prediction

`prediction = step(z)`

### Step 3 — Calculate error

`error = target - prediction`

If the prediction is already correct, the error is `0`.

### Step 4 — Update weights

`weights = weights + learning_rate * error * inputs`

### Step 5 — Update bias

`bias = bias + learning_rate * error`

This is the actual learning process.

The neuron does not memorize the AND table. It adjusts its parameters based on its mistakes.

In [6]:
# Reset parameters before training
weights = np.zeros(2)
bias = 0.0

for epoch in range(epochs):

    total_errors = 0

    for inputs, target in zip(X, y):

        # 1. Calculate weighted sum
        z = np.dot(inputs, weights) + bias

        # 2. Make prediction
        prediction = step_function(z)

        # 3. Calculate prediction error
        error = target - prediction

        # 4. Update weights
        weights += learning_rate * error * inputs

        # 5. Update bias
        bias += learning_rate * error

        # Count mistakes
        if error != 0:
            total_errors += 1

    print(
        f"Epoch {epoch + 1:2d} | "
        f"Weights: {weights} | "
        f"Bias: {bias:.2f} | "
        f"Errors: {total_errors}"
    )

Epoch  1 | Weights: [0.1 0.1] | Bias: 0.10 | Errors: 1
Epoch  2 | Weights: [0.2 0.1] | Bias: 0.00 | Errors: 3
Epoch  3 | Weights: [0.2 0.1] | Bias: -0.10 | Errors: 3
Epoch  4 | Weights: [0.2 0.2] | Bias: -0.10 | Errors: 2
Epoch  5 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 1
Epoch  6 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 0
Epoch  7 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 0
Epoch  8 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 0
Epoch  9 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 0
Epoch 10 | Weights: [0.2 0.1] | Bias: -0.20 | Errors: 0


## 7. What Happened During Training?

The important thing to notice is that the **weights and bias changed**.

For example, if the neuron predicts the wrong answer:

```text
target     = 1
prediction = 0

error = 1 - 0
      = 1
```

The weights are increased in the direction of the input.

If it predicts:

```text
target     = 0
prediction = 1

error = 0 - 1
      = -1
```

the weights are decreased.

Therefore, the perceptron gradually moves its decision boundary until it correctly separates the classes.

### Important concept

**Learning = repeatedly adjusting weights and bias based on prediction errors.**

## 8. Inspect the Learned Parameters

After training, let's see what the neuron learned.

In [7]:
print("Final learned weights:", weights)
print("Final learned bias:", bias)

Final learned weights: [0.2 0.1]
Final learned bias: -0.2


## 9. Test the Trained Neuron

Now we use the learned weights and bias to make predictions.

Notice that during testing we **do not update the weights**.

We only calculate:

`weighted sum → activation → prediction`

In [8]:
print("Input -> Prediction")

for inputs in X:
    z = np.dot(inputs, weights) + bias
    prediction = step_function(z)

    print(f"{inputs} -> {prediction}")

Input -> Prediction
[0 0] -> 0
[0 1] -> 0
[1 0] -> 0
[1 1] -> 1


## 10. Compare Predictions with Expected Values

Let's calculate the predictions and check whether the neuron learned the AND gate correctly.

In [9]:
predictions = []

for inputs in X:
    z = np.dot(inputs, weights) + bias
    prediction = step_function(z)
    predictions.append(prediction)

predictions = np.array(predictions)

print("Expected :", y)
print("Predicted:", predictions)

print("\nAll predictions correct:", np.array_equal(y, predictions))

Expected : [0 0 0 1]
Predicted: [0 0 0 1]

All predictions correct: True


## 11. Complete Neuron Logic

The entire perceptron can be summarized as:

```text
Inputs
  ↓
Multiply each input by its weight
  ↓
Add all weighted inputs
  ↓
Add bias
  ↓
Weighted sum (z)
  ↓
Step activation function
  ↓
Prediction
  ↓
Compare with target
  ↓
Calculate error
  ↓
Update weights and bias
  ↓
Repeat
```

Mathematically:

### Forward pass

`z = X·W + b`

`prediction = step(z)`

### Error

`error = target - prediction`

### Learning

`W = W + learning_rate × error × X`

`b = b + learning_rate × error`

This is the basic idea behind learning in a perceptron.

## 12. Try the OR Gate

The same neuron can learn the OR gate.

Only change the target values:

| x1 | x2 | Expected output |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 1 |

Run the following cell and reuse the same training logic.

In [10]:
# OR gate
X_or = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y_or = np.array([0, 1, 1, 1])

weights_or = np.zeros(2)
bias_or = 0.0

for epoch in range(epochs):
    for inputs, target in zip(X_or, y_or):

        z = np.dot(inputs, weights_or) + bias_or
        prediction = step_function(z)

        error = target - prediction

        weights_or += learning_rate * error * inputs
        bias_or += learning_rate * error

print("Learned weights:", weights_or)
print("Learned bias:", bias_or)

print("\nOR gate predictions:")

for inputs in X_or:
    z = np.dot(inputs, weights_or) + bias_or
    prediction = step_function(z)

    print(f"{inputs} -> {prediction}")

Learned weights: [0.1 0.1]
Learned bias: 0.0

OR gate predictions:
[0 0] -> 0
[0 1] -> 1
[1 0] -> 1
[1 1] -> 1


## 13. Why Can't One Perceptron Learn XOR?

Now try:

| x1 | x2 | XOR |
|---:|---:|----:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

XOR is **not linearly separable**.

A single perceptron creates only one linear decision boundary.

Therefore:

- AND → single perceptron can solve it
- OR → single perceptron can solve it
- XOR → single perceptron cannot solve it

This limitation is one reason neural networks use **multiple neurons and hidden layers**.

In [11]:
# XOR target values
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y_xor = np.array([0, 1, 1, 0])

print("XOR data:")
for inputs, target in zip(X_xor, y_xor):
    print(f"{inputs} -> {target}")

print("\nA single perceptron cannot correctly separate these four points.")

XOR data:
[0 0] -> 0
[0 1] -> 1
[1 0] -> 1
[1 1] -> 0

A single perceptron cannot correctly separate these four points.


# Key Takeaways

### What we built

We implemented a complete **single neuron/perceptron from scratch using NumPy**.

### The neuron has

- Inputs
- Weights
- Bias
- Weighted sum
- Activation function
- Prediction

### The learning process

The perceptron:

1. Takes input
2. Calculates weighted sum
3. Produces prediction
4. Compares prediction with target
5. Calculates error
6. Updates weights
7. Updates bias
8. Repeats

### Most important formula

`z = x·w + b`

### Learning rule

`w = w + learning_rate × error × x`

`b = b + learning_rate × error`

### Big picture

```text
Perceptron
    ↓
Multiple neurons
    ↓
Neural network
    ↓
Hidden layers
    ↓
Deep neural network
    ↓
Deep Learning
```

This exercise is the foundation for understanding what frameworks such as **TensorFlow/Keras** automate for us.